# 01_Theory: 서브워드 토크나이제이션 심층 분석 (BPE & WordPiece)

## 개요
LLM(대형 언어 모델)은 텍스트 자체를 이해하지 못하며, 오직 **숫자(Token ID)**만을 처리할 수 있습니다. 그렇다면 "machine learning"이라는 문장을 모델이 이해할 수 있는 숫자로 어떻게 변환할까요? 

과거에는 띄어쓰기 기준으로 단어를 쪼개거나(Word-level), 알파벳 하나하나를 쪼개는(Character-level) 방식을 사용했습니다. 하지만 현재 대다수의 SOTA(State-of-the-Art) 모델들은 **서브워드(Subword) 토크나이제이션** 방식을 사용합니다.

이 문서에서는 현대 NLP 토크나이제이션의 양대 산맥인 **BPE(Byte Pair Encoding)**와 **WordPiece**의 알고리즘 원리, 수학적 배경, 그리고 구체적인 작동 방식을 기존보다 훨씬 깊이 있게 파헤칩니다.

---
## 1. 토크나이제이션의 딜레마와 서브워드(Subword)의 등장

자연어를 쪼개는 기준(Unit)을 어떻게 설정하느냐에 따라 치명적인 트레이드오프(Trade-off)가 발생합니다.

### 1.1 단어 단위 (Word-level Tokenization)
* **방식**: `['machine', 'learning', 'is', 'great']`
* **장점**: 인간의 직관과 완벽히 일치하며, 토큰 하나하나가 독립적인 의미를 가집니다.
* **치명적 단점 (OOV 문제)**: `machine`, `machines`, `machinery`를 모두 다른 토큰으로 취급해야 합니다. 어휘 사전(Vocabulary)의 크기가 기하급수적으로 커지며, 사전에 없는 신조어나 오탈자가 등장하면 모델은 이를 `<UNK>` (Unknown) 토큰으로 처리해버려 정보 손실이 발생합니다.

### 1.2 문자 단위 (Character-level Tokenization)
* **방식**: `['m','a','c','h','i','n','e']`
* **장점**: 어휘 사전의 크기가 매우 작아집니다(영어 기준 알파벳+특수기호 약 100개 내외). OOV(Out-of-Vocabulary) 문제가 절대 발생하지 않습니다.
* **치명적 단점**: 시퀀스의 길이가 비정상적으로 길어져 모델의 연산량이 폭발합니다. 또한 'm'이나 'a' 자체에는 의미가 없으므로 모델이 단어의 의미를 학습하기가 매우 어려워집니다.

### 1.3 서브워드 단위 (Subword Tokenization)
* **핵심 철학**: "자주 등장하는 단어는 하나의 통째 토큰으로, 드물게 등장하는 단어는 더 작은 의미 단위(형태소, 어근 등)로 쪼개자!"
* **방식**: `['mach', '##ine']` (WordPiece 예시)
* **효과**: OOV 문제를 해결하면서도(최악의 경우 문자 단위로 쪼개짐), 시퀀스 길이와 어휘 사전 크기의 완벽한 균형을 이룹니다.

---
## 2. BPE (Byte Pair Encoding) 심층 분석

BPE는 1994년 원래 **데이터 압축(Data Compression)** 알고리즘으로 제안되었습니다. 이후 2016년 신경망 기계번역(NMT)에 도입되면서 서브워드 토크나이저의 대명사가 되었고, 현재 OpenAI의 **GPT 시리즈(GPT-2, GPT-3, GPT-4)**를 비롯한 대다수 LLM의 토크나이저 표준으로 자리 잡았습니다.

### 2.1 핵심 원리: "탐욕적 빈도 기반 병합 (Greedy Frequency Merge)"
BPE는 코퍼스(학습 데이터)에서 **가장 자주 같이 등장하는 글자(또는 바이트) 쌍을 반복적으로 병합**하여 하나의 덩어리(토큰)로 만듭니다. '탐욕적(Greedy)'이라는 말은 오직 '단순 빈도수'만을 기준으로 삼아 가장 높은 것을 맹목적으로 병합한다는 의미입니다.

### 2.2 수학적/논리적 작동 과정 (알고리즘)
1. **초기화**: 코퍼스 내의 모든 단어를 글자(Character) 단위로 산산조각 냅니다. 단어의 끝을 표시하기 위해 빈도가 높은 단어가 다른 단어의 접두사로 합쳐지는 것을 방지하는 `</w>` 특수 기호를 붙입니다.
2. **쌍(Pair) 통계 계산**: 쪼개진 글자들 중에서 서로 인접한 두 토큰 쌍의 등장 빈도를 모두 카운트합니다.
3. **최다 빈도 쌍 병합**: 빈도가 가장 높은 쌍을 찾아 하나의 새로운 토큰으로 병합(Merge)합니다.
4. **반복**: 목표로 설정해둔 어휘 사전 크기(Vocabulary Size, 예: 50,000개)에 도달할 때까지 2~3번 과정을 무한히 반복합니다.

### 2.3 BPE 병합(Merge) 상세 시뮬레이션
다음과 같은 단어 코퍼스가 주어졌다고 가정해 봅시다. 
`(빈도수) 단어`
* (5) `l o w </w>`
* (2) `l o w e s t </w>`
* (6) `n e w e r </w>`
* (3) `w i d e r </w>`

**[초기 상태 어휘 사전]**: `l`, `o`, `w`, `e`, `s`, `t`, `n`, `r`, `i`, `d`, `</w>`

**▶ 1회차 병합:** 전체 코퍼스에서 인접한 두 토큰의 쌍을 셉니다.
* `e`와 `r`이 붙어있는 빈도 = 6 (`newer`) + 3 (`wider`) = **9회**
* `e`와 `w`가 붙어있는 빈도 = 6 (`newer`) = 6회
가장 빈도가 높은 `e`와 `r`을 `er`이라는 하나의 토큰으로 병합합니다.
* 코퍼스 업데이트: `n e w er </w>`, `w i d er </w>`
* 어휘 사전 추가: `er`

**▶ 2회차 병합:**
* `er`과 `</w>`가 붙어있는 빈도 = 6 (`newer`) + 3 (`wider`) = **9회**
가장 빈도가 높은 `er`과 `</w>`를 `er</w>`로 병합합니다.
* 코퍼스 업데이트: `n e w er</w>`, `w i d er</w>`
* 어휘 사전 추가: `er</w>`

**▶ 3회차 병합:**
* `l`과 `o`가 붙어있는 빈도 = 5 (`low`) + 2 (`lowest`) = **7회**
* `e`와 `s`가 붙어있는 빈도 = 2 (`lowest`) = 2회
`l`과 `o`를 `lo`로 병합합니다.
* 코퍼스 업데이트: `lo w </w>`, `lo w e s t </w>`
* 어휘 사전 추가: `lo`

**[결과 해석]**: 이 과정을 목표 크기만큼 반복하면, 자주 쓰이는 `low`나 `newer`는 통째로 하나의 토큰 `low</w>`가 됩니다. 반면 빈도가 적은 `lowest`는 `low`와 `est</w>`라는 두 개의 서브워드로 분리되어 남게 됩니다. 이를 통해 미등록 단어(OOV)라도 기존에 학습된 서브워드의 조합으로 표현할 수 있게 됩니다.

### 2.4 BPE의 한계점 (다국어 환경에서의 의미 파편화)
BPE는 철저히 **'단순 등장 빈도'** 중심입니다. 따라서 언어학적인 형태소(어근, 접두사, 접미사)를 전혀 고려하지 않습니다.
특히 영문 위주로 학습된 GPT 모델에 한글을 넣으면 치명적인 문제가 발생합니다. 한글은 영문에 비해 빈도수가 낮아 병합의 우선순위에서 밀리기 때문에, 글자 단위 혹은 자모음 수준까지 무의미하게 산산조각(예: 꿀벌 -> ㄲ, ㅜ, ㄹ, ㅂ, ㅓ, ㄹ)이 나버리는 현상이 발생합니다. 이는 시퀀스 길이를 증가시켜 API 비용 증가와 모델 성능 하락을 초래합니다.

---
## 3. WordPiece 토크나이제이션 심층 분석

WordPiece는 구글이 2016년 구글 번역기(GNMT) 내부 시스템을 위해 개발했고, 2018년 전설적인 언어 모델인 **BERT(Bidirectional Encoder Representations from Transformers)**에 적용되면서 세상에 널리 알려졌습니다. 
겉보기에는 BPE와 매우 유사하게 작동하지만, 내부적으로 병합 대상을 결정하는 **수학적 기준(Scoring Method)**이 완전히 다릅니다.

### 3.1 핵심 원리: "우도(Likelihood)와 정보 이득(Information Gain) 극대화"
BPE가 "가장 많이 나타나는 쌍"을 맹목적으로 합친다면, WordPiece는 **"이 두 개를 합쳤을 때, 전체 언어 모델의 언어 데이터 표현 확률(Likelihood)이 얼마나 상승하는가?"**를 철저히 계산합니다.

즉, 단순히 두 토큰이 자주 같이 나온다고 무조건 합치는 것이 아닙니다. 두 토큰이 각각 따로 쓰일 때의 확률보다, **둘이 함께 결합되어 쓰일 때의 확률이 압도적으로 높을 때(상호 의존성이 높을 때)**만 병합을 수행합니다.

### 3.2 수학적 결정 공식 (Scoring)
두 토큰 $a$와 $b$를 병합하여 새로운 토큰 $ab$를 만들지 결정하는 점수(Score)는 다음과 같이 계산됩니다.

$$ Score = rac{P(ab)}{P(a) 	imes P(b)} $$

* $P(a)$: 코퍼스 전체에서 토큰 $a$가 단독으로 등장할 확률 (개별 빈도 / 전체 토큰 수)
* $P(b)$: 코퍼스 전체에서 토큰 $b$가 단독으로 등장할 확률
* $P(ab)$: 토큰 $a$와 $b$가 연달아 등장할 확률
* **공식 해석**: 분모 $P(a) 	imes P(b)$는 $a$와 $b$가 완전히 독립적일 때 우연히 같이 등장할 확률입니다. 분자 $P(ab)$는 실제로 같이 등장한 확률입니다. 
  만약 'a'와 'the'처럼 각각 자주 쓰이는 단어라면 $P(a)$와 $P(b)$가 매우 큽니다. 따라서 아무리 둘이 자주 붙어있어 $P(ab)$가 높다 하더라도 분모가 너무 크기 때문에 Score는 낮게 나옵니다.
  반면, `##able`이나 `##ing`처럼 특정 어근 뒤에 종속적으로만 등장하는 접미사의 경우 점수가 매우 높게 나옵니다.

### 3.3 구현적 특징: 접두사 `##` (서브워드 표시)
WordPiece의 가장 큰 시각적 특징은 단어의 첫 번째 조각(Subword)을 제외한 **나머지 조각들의 앞에 `##` 기호를 붙인다**는 점입니다.

* 단어 분할 예시: `unbelievable` → `['un', '##bel', '##ieva', '##ble']`

**왜 `##`을 사용할까요? (디코딩의 완벽성)**
`##` 기호는 나중에 모델이 출력한 토큰들을 다시 완전한 문장으로 복원(Decoding)할 때 핵심적인 역할을 합니다.
만약 `['I', 'love', 'machine', 'learn', '##ing']` 이라는 토큰들이 있다면:
1. `##`이 없는 토큰은 그 앞에 **띄어쓰기(공백)**를 추가합니다.
2. `##`이 있는 토큰은 공백 없이 바로 앞 토큰에 **이어 붙입니다**.
이 간단한 규칙만으로 원래 문장의 띄어쓰기를 100% 완벽하게 복구할 수 있습니다.

### 3.4 왜 BERT는 WordPiece를 선택했을까?
BERT는 문맥(Context)을 양방향에서 깊게 이해해야 하는 자연어 '이해(NLU)' 특화 모델입니다. BPE처럼 기계적으로 바이트를 묶는 것보다, WordPiece처럼 **통계적 상호 의존도**를 고려해 쪼개는 것이 영어의 실제 형태소(어근, 접두/접미사) 분할과 훨씬 유사한 결과를 냅니다. 이는 모델이 문장 내 단어들의 관계성을 더 정교하게 파악하는 데 큰 도움을 줍니다.

---
## 4. 핵심 비교 요약 (BPE vs WordPiece)

| 항목 | BPE (Byte Pair Encoding) | WordPiece |
|------|-------------------------|-----------|
| **병합 우선순위 기준** | **단순 등장 빈도 (Frequency)**<br>(가장 많이 같이 나온 쌍부터 병합) | **정보 이득 및 우도 (Likelihood)**<br>상호 의존성이 높은 쌍부터 병합 |
| **디코딩(복원) 기호** | 단어 끝에 `</w>` 또는 공백 위치에 `Ġ` | 서브워드 접두사로 `##` 사용 |
| **형태소 분할 성향** | 기계적이고 투박하게 잘림 (빈도 의존적) | 상대적으로 언어학적 형태소 구조에 가깝게 잘림 |
| **주요 채택 언어 모델** | **OpenAI GPT 시리즈**, RoBERTa, LLaMA(일부) | **Google BERT**, Electra, DistilBERT |
| **특징** | 압축 알고리즘 출신으로 매우 빠르고 단순하여 초대규모 코퍼스에 적합함 | 우도를 계산해야 하므로 어휘 사전 구축 시 연산량이 더 많음 |

## 결론
BPE와 WordPiece 모두 **"OOV 문제를 해결하고 어휘 사전 크기를 최적화한다"**는 동일한 목표를 달성하기 위한 서브워드 토크나이저입니다. 
하지만 GPT와 같은 대규모 텍스트 생성 모델(LLM)은 계산이 단순하고 빠르며 바이트 수준의 확장이 용이한 **BPE**를 선호하는 반면, 
정밀한 문맥 이해가 필수적인 BERT 등은 통계적으로 유의미한 형태소 분할을 지향하는 **WordPiece**를 채택했습니다. 
프로젝트를 진행할 때 자신이 사용하는 모델이 어떤 토크나이저를 쓰는지 정확히 이해하는 것이 프롬프트 최적화와 파인튜닝의 첫걸음입니다.